In [16]:
# 1. 필수 라이브러리 설치
!pip install -q transformers torch pandas emoji soynlp

from transformers import pipeline
import torch
import pandas as pd
import re

# 2. 모델 로드 (sentiment_model.md에서 권장하는 구어체 특화 모델)
device = 0 if torch.cuda.is_available() else -1
model_path = "Copycats/koelectra-base-v3-generalized-sentiment-analysis"
# model_path = "nlptown/bert-base-multilingual-uncased-sentiment"
# model_path = "monologg/koelectra-base-v3-discriminator"
# model_path = "monologg/koelectra-base-v3-discriminator"
print("모델을 로드 중입니다. 잠시만 기다려 주세요...")
classifier = pipeline("sentiment-analysis", model=model_path, device=device)
print("모델 로드 완료!")


모델을 로드 중입니다. 잠시만 기다려 주세요...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4112.64it/s]
ElectraForSequenceClassification LOAD REPORT from: Copycats/koelectra-base-v3-generalized-sentiment-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
electra.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


모델 로드 완료!


In [8]:
# 1. 6대 속성별 키워드 사전 (sentiment_model.md 내용 기반)
aspect_keywords = {
    "기호성": ["먹어", "먹는", "순삭", "기호성", "입맛", "뇸뇸", "거부", "식탐", "안 먹", "잘 먹"],
    "생체반응": ["눈물", "가려움", "긁어", "털", "모질", "피부", "활력", "알러지", "알레르기"],
    "소화/배변": ["설사", "묽은", "변", "황금변", "맛동산", "응가", "똥", "구토", "토해", "소화"],
    "제품 성상": ["알갱이", "크기", "사이즈", "키블", "가루", "단단", "딱딱", "부드러운", "성상"],
    "성분/원료": ["원료", "성분", "가수분해", "그레인프리", "육류", "함량", "원산지", "첨가물"],
    "냄새": ["냄새", "향", "구려", "꼬순내", "비린내", "역해", "악취"]
}

# 2. 문장 분리 함수
def split_sentences(text):
    return [s.strip() for s in re.split(r'[.!?]\\s*', text) if s.strip()]

# 3. 속성 매핑 함수
def get_aspects(sentence):
    found_aspects = []
    for aspect, keywords in aspect_keywords.items():
        if any(keyword in sentence for keyword in keywords):
            found_aspects.append(aspect)
    return found_aspects


In [17]:
# 테스트용 펫 리뷰 샘플 (복합적인 감정이 섞인 리뷰)
test_reviews = [
    "사료 알갱이가 작아서 노령견인 우리 애가 먹기 편해해요. 그런데 냄새가 좀 역하고, 먹고 나서 바로 설사를 했어요.",
    "기호성은 좋은데 눈물이 너무 많이 나요. 알러지 성분이 의심되네요.",
    "성분은 믿음직스러운데 애가 입도 안 대요. 냄새는 고소해서 좋은데 아쉽네요.",
    "황금변을 봐서 너무 좋아요! 모질도 윤기가 자르르 흐르네요. 대만족입니다."
]

# 분석 결과를 담을 리스트
structured_data = []
all_aspects = list(aspect_keywords.keys())

for review in test_reviews:
    sentences = split_sentences(review)
    for sent in sentences:
        # 이 문장에서 언급된 속성들을 찾습니다.
        found_aspects = get_aspects(sent)
        
        if found_aspects:
            # 감성 분석 수행 (1=긍정, 0=부정)
            res = classifier(sent)[0]
            sentiment_label = "긍정" if res['label'] == '1' else "부정"
            score = res['score']
            
            # 기본 행 데이터 구성 (문장 원본 포함)
            row = {"문장": sent}
            
            # 모든 속성 컬럼을 돌면서, 이번 문장에 해당하면 감성 결과 기록
            for asp in all_aspects:
                if asp in found_aspects:
                    row[asp] = sentiment_label
                else:
                    row[asp] = "-"  # 해당 문장에 해당 속성이 없는 경우
            
            row["종합 확신도"] = f"{score:.2f}"
            structured_data.append(row)

# 결과 데이터프레임 생성
df_final = pd.DataFrame(structured_data)

# 보기 좋게 컬럼 순서 조정 (문장 -> 속성들 -> 확신도)
ordered_columns = ["문장"] + all_aspects + ["종합 확신도"]
df_final = df_final[ordered_columns]

# 긍정/부정 텍스트에 색상을 입히는 스타일 함수
def style_sentiment(val):
    if val == "긍정": return 'color: blue; font-weight: bold'
    if val == "부정": return 'color: red; font-weight: bold'
    return 'color: #999' # '-' 부분은 흐릿하게

# 결과 출력 (최신 Pandas map 사용)
try:
    styled_df = df_final.style.map(style_sentiment, subset=all_aspects)
except AttributeError:
    styled_df = df_final.style.applymap(style_sentiment, subset=all_aspects)

styled_df




,문장,기호성,생체반응,소화/배변,제품 성상,성분/원료,냄새,종합 확신도
0,"사료 알갱이가 작아서 노령견인 우리 애가 먹기 편해해요. 그런데 냄새가 좀 역하고, 먹고 나서 바로 설사를 했어요.",-,-,부정,부정,-,부정,0.86
1,기호성은 좋은데 눈물이 너무 많이 나요. 알러지 성분이 의심되네요.,부정,부정,-,-,부정,-,0.94
2,성분은 믿음직스러운데 애가 입도 안 대요. 냄새는 고소해서 좋은데 아쉽네요.,-,-,-,-,부정,부정,0.96
3,황금변을 봐서 너무 좋아요! 모질도 윤기가 자르르 흐르네요. 대만족입니다.,-,긍정,긍정,-,-,-,0.99
